# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata fields
meta = dataset.metadata

print(f"Dataset title: {getattr(meta, 'name', '')}")
print(f"Description: {getattr(meta, 'description', '')}")


## 2. Data Overview

Review available record sets, fields, and their `@id`s (unique IDs in the Croissant schema). This helps you understand which tables or file-like structures the dataset contains, and what columns or fields are present.

In [ ]:
print("Available record sets (by @id):")
record_sets = [rs['@id'] for rs in getattr(meta, 'record_sets', [])] if hasattr(meta, 'record_sets') else []

# If `record_sets` is empty, attempt to infer record sets via the public API
if not record_sets:
    # Try .record_sets() if available
    try:
        record_sets = [rs['@id'] for rs in dataset.record_sets()]
    except Exception:
        # Fallback: try to scan the metadata
        if hasattr(meta, 'recordSet'):
            recs = getattr(meta, 'recordSet')
            if isinstance(recs, list):
                record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in recs]
            elif isinstance(recs, dict) and '@id' in recs:
                record_sets = [recs['@id']]
        else:
            record_sets = []

if not record_sets:
    print("No record sets found in the metadata. Please inspect the raw JSON-LD for deep structure or check mlcroissant version.")
else:
    for i, rs_id in enumerate(record_sets):
        print(f"  {i+1}. {rs_id}")

# For each record set, try printing field IDs and column names
print('\nField IDs and example data for each record set (using their @id):')
for rs_id in record_sets:
    print(f"Record set: {rs_id}")
    try:
        # List fields
        fields = dataset.fields(record_set=rs_id)
        field_ids = [f['@id'] for f in fields]
        print(f"  Fields: {field_ids}")
        # Show a sample record
        records_iter = dataset.records(record_set=rs_id)
        example = next(records_iter)
        print(f"  Example record keys: {list(example.keys())}")
    except Exception as e:
        print(f"  Could not extract fields or record example ({e})")
    print('-' * 40)


## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll select the first available record set for demonstration.

In [ ]:
# Use the first record set found as main data table
if not record_sets:
    raise ValueError('No record sets available to extract.')

main_record_set_id = record_sets[0]
print(f"Selected record set: {main_record_set_id}")

# Optionally: list ALL record set ids you want to extract. Here just use all found.
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records for record set {rs_id}")
    dataframes[rs_id] = df

# Show columns of the main data table
print(f"Columns in {main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())

# Show top 5 records
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.


In [ ]:
df = dataframes[main_record_set_id]
print("Column data types detected:")
print(df.dtypes)

# Try to choose a numeric field by heuristic
numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
if not numeric_candidates:
    # Try to coerce columns to numeric to find candidates
    for c in df.columns:
        coerced = pd.to_numeric(df[c], errors='coerce')
        if coerced.notnull().sum() > 0:
            df[c] = coerced
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]

if not numeric_candidates:
    raise ValueError("No numeric columns found for analysis.")

numeric_field = numeric_candidates[0]
print(f"Selected numeric field for analysis: {numeric_field}")

# Example filtering: keep only records where numeric value > threshold
threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} rows")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to choose a categorical/group-by field (not numeric)
group_candidates = [c for c in df.columns if c != numeric_field and not pd.api.types.is_numeric_dtype(df[c])]
if group_candidates:
    group_field = group_candidates[0]
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")


## 5. Visualization

Visualize distributions or relationships between fields in the dataset. (e.g. histogram of the numeric field, boxplot by group field).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f"Histogram of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If grouping field is present, show boxplot
if group_candidates:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=30)
    plt.show()


## 6. Conclusion

In this notebook, we loaded and explored the FAIR^2 dataset on mass spectrometry proteomics of extracellular vesicles from human mast cells, using the [mlcroissant](https://github.com/mlcommons/croissant) library. We examined dataset structure and fields, extracted tabular data, applied basic EDA and visualization. This workflow can be adapted for more advanced analyses, including data cleaning, hypothesis testing, or machine learning pipelines, leveraging the full Croissant schema details such as `@id` for precise integration.
